# Synthetic Data Generator — Privacy-Safe ML Benchmark

**Domain**: Financial fraud detection  
**Dataset**: ULB Credit Card Fraud Detection (Kaggle)  
**Generators**: Gaussian Copula · CTGAN · TVAE  
**Evaluation axes**: Realism · Downstream Utility · Privacy Leakage  

---

## Pipeline overview

```
Raw CSV  →  Profile  →  Preprocess  →  Train generators  →  Generate
                                                               │
                                          ┌────────────────────┤
                                          │                    │
                                     Realism eval       Utility eval
                                     (KS, Wasserstein,  (AUROC, F1 on
                                      corr dist,         real test set)
                                      discriminator)
                                          │
                                    Privacy checks
                                    (dup rate, NN dist,
                                     rare memorisation)
```

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root to path when running from notebooks/

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Load config
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded.')
print(f"  Dataset: {cfg['data']['dataset']}")
print(f"  Target column: {cfg['data']['target_column']}")

---
## 1. Data Loading and Profiling

**Download the dataset first:**
```
# From Kaggle CLI:
kaggle datasets download -d mlg-ulb/creditcardfraud
unzip creditcardfraud.zip -d ../data/raw/
```
Or download manually from https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud and place `creditcard.csv` in `data/raw/`.

In [ ]:
from src.preprocessing.preprocess import load_raw, profile

raw_path = Path('../') / cfg['data']['raw_path']
target_col = cfg['data']['target_column']

df = load_raw(raw_path)
data_card = profile(df, target_col=target_col)

df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df[target_col].value_counts()
axes[0].bar(['Legit (0)', 'Fraud (1)'], counts.values, color=['steelblue', 'firebrick'])
axes[0].set_title('Class Distribution (raw counts)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(counts.values, labels=['Legit', 'Fraud'], colors=['steelblue', 'firebrick'],
            autopct='%1.3f%%', startangle=90)
axes[1].set_title('Class Balance')

plt.suptitle('ULB Credit Card Fraud Dataset — Class Distribution', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Fraud rate: {data_card['fraud_rate_pct']:.4f}% — severe class imbalance")

In [ ]:
# Feature distributions (sample of PCA-transformed columns V1-V6)
sample_cols = [c for c in df.columns if c.startswith('V')][:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

import seaborn as sns
for ax, col in zip(axes, sample_cols):
    sns.kdeplot(df[df[target_col]==0][col], ax=ax, label='Legit', color='steelblue', fill=True, alpha=0.4)
    sns.kdeplot(df[df[target_col]==1][col], ax=ax, label='Fraud', color='firebrick', fill=True, alpha=0.4)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Legit vs Fraud', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/feature_distributions_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Preprocessing

In [ ]:
from src.preprocessing.preprocess import run_preprocessing

train, val, test, preprocessor, data_card = run_preprocessing(
    raw_path=raw_path,
    output_dir=Path('../') / cfg['data']['processed_dir'],
    target_col=target_col,
    train_frac=cfg['splits']['train'],
    val_frac=cfg['splits']['val'],
    random_state=cfg['splits']['random_state'],
)

print(f"\nSplit sizes — train: {len(train):,}  val: {len(val):,}  test: {len(test):,}")
print(f"Test set is held out (real only) for all downstream evaluation.")

---
## 3. Train Synthetic Data Generators

We train three models:
- **Gaussian Copula** — classical statistical baseline (fast)
- **CTGAN** — GAN-based tabular generator
- **TVAE** — VAE-based tabular generator

> Training CTGAN and TVAE takes ~10-20 minutes each on CPU. Reduce `epochs` in `config.yaml` for a faster demo run.

In [ ]:
from src.generators.train_generators import run_training

synth_datasets = run_training(
    train_df=train,
    model_dir=Path('../outputs/models'),
    synthetic_dir=Path('../') / cfg['data']['synthetic_dir'],
    target_col=target_col,
    ctgan_epochs=cfg['generators']['ctgan']['epochs'],
    tvae_epochs=cfg['generators']['tvae']['epochs'],
    batch_size=cfg['generators']['ctgan']['batch_size'],
)

for name, sdf in synth_datasets.items():
    fraud_rate = sdf[target_col].astype(float).mean() * 100
    print(f"  {name}: {len(sdf):,} rows  fraud_rate={fraud_rate:.4f}%")

---
## 4. Realism Evaluation

In [ ]:
from src.evaluation.realism import (
    realism_scorecard, plot_distributions,
    plot_correlation_heatmaps, plot_pca_overlap
)

reports_dir = Path('../reports')
reports_dir.mkdir(exist_ok=True)

real_scorecard = realism_scorecard(train, synth_datasets, target_col=target_col)
real_scorecard

In [ ]:
plot_distributions(train, synth_datasets, target_col=target_col, n_cols_to_plot=6, output_dir=reports_dir)

In [ ]:
plot_correlation_heatmaps(train, synth_datasets, target_col=target_col, output_dir=reports_dir)

In [ ]:
plot_pca_overlap(train, synth_datasets, target_col=target_col, n_samples=3000, output_dir=reports_dir)

### Realism Interpretation

| Metric | Ideal value | What it means |
|---|---|---|
| KS statistic | → 0 | Per-column distribution matches real data |
| Wasserstein distance | → 0 | Earth-mover distance between distributions |
| Correlation distance | → 0 | Feature correlations preserved |
| Discriminator AUROC | → 0.5 | Classifier cannot distinguish real vs synthetic |

---
## 5. Downstream ML Utility Evaluation

All models are evaluated on the **same held-out real test set**.

In [ ]:
from src.evaluation.utility import (
    run_utility_benchmark, plot_utility_comparison, plot_metric_heatmap
)

utility_results = run_utility_benchmark(
    train_real=train,
    test_real=test,
    synth_datasets=synth_datasets,
    target_col=target_col,
    aug_ratios=cfg['evaluation']['aug_ratios'],
    random_state=cfg['splits']['random_state'],
)

utility_results

In [ ]:
plot_utility_comparison(utility_results, metric='auroc', output_dir=reports_dir)

In [ ]:
plot_utility_comparison(utility_results, metric='auprc', output_dir=reports_dir)

In [ ]:
plot_metric_heatmap(utility_results, output_dir=reports_dir)

### Utility Interpretation

- **Real-only** is the gold standard. If synthetic-only is close, the generator is useful.
- **Mixed (real + synthetic)** can improve recall on minority classes via augmentation.
- **AUPRC** is the most informative metric for imbalanced fraud detection.

---
## 6. Privacy Leakage Analysis

In [ ]:
from src.privacy.privacy_checks import (
    privacy_scorecard, plot_nn_distance_distributions
)

priv_scorecard = privacy_scorecard(
    train, synth_datasets,
    target_col=target_col,
    n_neighbors=cfg['privacy']['n_neighbors'],
)

priv_scorecard

In [ ]:
plot_nn_distance_distributions(train, synth_datasets, target_col=target_col, output_dir=reports_dir)

### Privacy Interpretation

| Metric | Safe range | Risk indicator |
|---|---|---|
| Exact duplicate rate | ≈ 0% | > 0.1% is a red flag |
| NN mean distance | > baseline | Much lower than real→real = memorisation |
| Rare record memorisation | ≈ 0% | Any near-zero distance to minority rows is high risk |

---
## 7. Final Scorecard Summary

In [ ]:
import json

# Best XGBoost utility per training strategy
xgb_utility = utility_results[utility_results['model'] == 'xgboost'].copy()
xgb_utility = xgb_utility.sort_values('auroc', ascending=False)

print('\n=== TOP UTILITY RESULTS (XGBoost, sorted by AUROC) ===')
print(xgb_utility[['training_data','auroc','auprc','f1','recall']].to_string(index=False))

print('\n=== REALISM SCORECARD ===')
print(real_scorecard.to_string(index=False))

print('\n=== PRIVACY SCORECARD ===')
print(priv_scorecard.to_string(index=False))

In [ ]:
# Save combined summary
summary = {
    'data_card': data_card,
    'realism': real_scorecard.to_dict(orient='records'),
    'utility_top5': xgb_utility.head(5).to_dict(orient='records'),
    'privacy': priv_scorecard.to_dict(orient='records'),
}

with open(reports_dir / 'pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Save CSVs
real_scorecard.to_csv(reports_dir / 'realism_scorecard.csv', index=False)
utility_results.to_csv(reports_dir / 'utility_results.csv', index=False)
priv_scorecard.to_csv(reports_dir / 'privacy_scorecard.csv', index=False)

print(f'Reports saved to {reports_dir.resolve()}')

---
## 8. AWS SageMaker / S3 Integration (Optional)

Use this section to upload data and reports to S3 for scalable storage and sharing.

In [ ]:
# Uncomment to provision buckets and upload artifacts

# from src.utils.s3_utils import provision_buckets, upload_directory

# provision_buckets(cfg)

# upload_directory(
#     local_dir='../reports',
#     bucket=cfg['aws']['buckets']['reports'],
#     s3_prefix=f"{cfg['aws']['prefix']}/reports",
#     region=cfg['aws']['region'],
# )

# upload_directory(
#     local_dir='../data/synthetic',
#     bucket=cfg['aws']['buckets']['synthetic'],
#     s3_prefix=f"{cfg['aws']['prefix']}/synthetic",
#     region=cfg['aws']['region'],
# )

print('S3 upload section — uncomment the lines above to enable.')

---

## Summary

This notebook demonstrates a full synthetic tabular data pipeline:

1. **Data**: ULB Credit Card Fraud — 284k rows, 0.17% fraud rate, PCA-transformed features.
2. **Generators**: Gaussian Copula (fast baseline), CTGAN (GAN-based), TVAE (VAE-based).
3. **Realism**: KS test, Wasserstein distance, correlation matrix comparison, real-vs-synthetic discriminator.
4. **Utility**: Fraud classifiers (XGBoost, RF, LR) trained on real-only, synthetic-only, and mixed data — all evaluated on held-out real test set.
5. **Privacy**: Exact duplicate rate, nearest-neighbour distance distribution, rare record memorisation check.

**Key finding**: Synthetic data alone is weaker than real data for fraud detection, but mixed training (real + synthetic augmentation) can improve recall on the minority fraud class — especially useful when sharing data across teams without exposing raw cardholder records.

**AWS path to production**:
- Store datasets in S3 (raw / curated / synthetic / reports buckets)
- Run generator training as SageMaker Training Jobs (parallelise across generators)
- Track experiments with SageMaker Experiments
- Schedule weekly synthetic data refresh via SageMaker Pipelines